![Cutie Hare](https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1452px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?20170313021652)


This file is made available under the Creative Commons CC0 1.0 Universal Public Domain Dedication.
The person who associated a work with this deed has dedicated the work to the public domain by waiving all of their rights to the work worldwide under copyright law, including all related and neighboring rights, to the extent allowed by law. You can copy, modify, distribute and perform the work, even for commercial purposes, all without asking permission.

In [115]:
import pandas as pd
import numpy as np

In [116]:
URL = "https://pasta.lternet.edu/package/data/eml/knb-lter-bnz/55/22/f01f5d71be949b8c700b6ecd1c42c701"
hares = pd.read_csv(URL)

In [117]:
hares.dtypes

date           object
time           object
grid           object
trap           object
l_ear          object
r_ear          object
sex            object
age            object
weight        float64
hindft        float64
notes          object
b_key         float64
session_id      int64
study          object
dtype: object

In [118]:
hares.head()
hares.shape
hares.isnull().sum()
hares['weight'].max() # 2365 
hares['weight'].min() # 0 
hares['hindft'].max() # 160
hares['hindft'].min() #60

60.0

In [119]:
# change the place where weight = 0 to an NA 
hares[hares['weight'] == 0] = np.nan

In [120]:
hares['age'].unique()

array([nan, 'J', 'A', 'a 1 yr.', 'a 3/4 yr.', 'a 1 yr', '1 yr', '1 yr.',
       '2 yrs.', '2 yrs', 'a 2 yrs.', '2.25 yrs', '3.5 yrs.', '3 yrs.',
       '2.5 yrs', '3.25 yrs.', 'A 1.5', '?', 'U', 'j', 'a', 'u', 'J 3/4',
       'A 3/4', 'A 1/2', '3/4/2013', '1/4/2013', '1/2/2013', '1', '1.25',
       '1.5'], dtype=object)

In [121]:
hares['sex'].unique()

array([nan, 'M', 'F', '?', 'F?', 'M?', 'pf', 'm', 'f', 'f?', 'm?', 'f ',
       'm '], dtype=object)

### Exploratory Question 
Why is the age and sex columns so messed up? 

## Detecting messy values

### Table for allows sex values 

| Syntax      | Description   |
| ----------- | ------------- |
| m           | male          |
| f           | female        |
| m?          | Not confirmed |

In [122]:
hares['sex'].value_counts().unique

<bound method Series.unique of sex
F     1161
M      730
f      556
m      515
?       40
F?      10
f        4
m        4
f?       3
M?       2
m?       2
pf       1
Name: count, dtype: int64>

In [123]:
hares['sex'].value_counts(dropna = False).unique

<bound method Series.unique of sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64>

### Discuss 
This column sucks. Citizian science = dif people recording data. People need a standarized way of discovering and inputing data. 

### e 
Theres whitespace! Need to strip

## Cleaning values 

In [124]:
hares['sex_simple'] = hares['sex'].str.strip().str.lower()
hares[(hares['sex_simple'] == "?") | (hares['sex_simple'] == "pf") | (hares['sex_simple'] == "f?") ] = np.nan


In [127]:
hares['sex_simple'].value_counts(dropna = False).unique


<bound method Series.unique of sex_simple
f      1721
m      1249
NaN     406
m?        4
Name: count, dtype: int64>

In [132]:
# How the presenter did this 

condition_list = [x.isin(['M', 'm', 'm_']), x.isin(['F','f','f_'])]
output_list = ['male', 'female']

new_column = np.select(condition_list, output_list, default = np.nan)
# np.select(inputlist, if_input)

hares['sex_simple']= new_column 

NameError: name 'x' is not defined

## Calculate mean weight 

In [128]:
hares.head()

,date,time,grid,trap,l_ear,r_ear,sex,age,weight,hindft,notes,b_key,session_id,study,sex_simple
0,11/26/1998,NaN,bonrip,1A,414D096A08,NaN,NaN,NaN,1370.0,160.0,NaN,917.0,51.0,Population,NaN
1,11/26/1998,NaN,bonrip,2C,414D320671,NaN,M,NaN,1430.0,NaN,NaN,936.0,51.0,Population,m
2,11/26/1998,NaN,bonrip,2D,414D103E3A,NaN,M,NaN,1430.0,NaN,NaN,921.0,51.0,Population,m
3,11/26/1998,NaN,bonrip,2E,414D262D43,NaN,NaN,NaN,1490.0,135.0,NaN,931.0,51.0,Population,NaN
4,11/26/1998,NaN,bonrip,3B,414D2B4B58,NaN,NaN,NaN,1710.0,150.0,NaN,933.0,51.0,Population,NaN


In [131]:
# Use groupby to calculate the mean weight by sex 
hares.groupby('sex_simple')['weight'].mean()

sex_simple
f     1365.164792
m     1349.935542
m?    1446.666667
Name: weight, dtype: float64